# 56. Sharing one `Parameter` object across components

**Objectives:**

- Reuse the exact same `Parameter` instance (a Blatt-Weisskopf `parent_radius`) as a field on
  two different `Resonance` components.
- Confirm `DecayModel` deduplicates it: it appears exactly once in `model.parameters`.
- Show what happens when two *different* `Parameter` objects accidentally share a `name` but
  disagree on `value`/`bounds` -- `DecayModel` must reject this rather than silently pick one.

Run the cells in order in a fresh kernel. Masses are in GeV, invariants in GeV^2.

In [1]:
from dalitzplotfitter import enable_x64
enable_x64()  # Must precede any numerical work: amplitudes use complex128.

from dalitzplotfitter import DecayChannel, DecayModel, NonResonant, Parameter, RealImag, Resonance

## 1. One `Parameter` instance, two components

`Resonance.parent_radius` accepts either a plain float or a `Parameter`. A plain
`Parameter(name, value, bounds=...)` (default `kind=ParameterKind.OTHER`, `owner=None`) is not
subject to the `owner`-matching checks that `Parameter.dynamics`/`Parameter.coefficient`
enforce for `ParameterKind.DYNAMICS`/`COEFFICIENT` parameters (see `decay.py`'s
`_validate_parameters`), so the very same instance can be handed to unrelated components
directly. Here `rho` and `f0` -- two otherwise independent resonances -- share one
`parent_radius` Parameter, e.g. because both are believed to see the same parent
Blatt-Weisskopf barrier.

In [2]:
shared_radius = Parameter("parent_radius", 5.0, bounds=(3.0, 7.0))

channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))
model = DecayModel(
    channel,
    components=[
        Resonance("rho", (0, 1), RealImag(1.0, 0.0), mass=0.7753, width=0.1491, spin=1,
                  parent_radius=shared_radius),
        Resonance("f0", (0, 1), RealImag(0.5, -0.2), mass=0.980, width=0.060, spin=0,
                  parent_radius=shared_radius),
        NonResonant(RealImag(0.3, 0.1), name="NR"),
    ],
    normalization_method="square-dalitz", normalization_resolution=40,
)
names = [p.name for p in model.parameters]
print("model.parameters:", names)
assert names.count("parent_radius") == 1
print("parent_radius appears exactly once, despite being referenced by two components.")

model.parameters: ['parent_radius']
parent_radius appears exactly once, despite being referenced by two components.


## 2. Two *different* `Parameter` objects, same name

If instead two separately-constructed `Parameter` objects happen to share a `name` but disagree
on `value` (or `bounds`, `fixed`, ...), `DecayModel._validate_parameters` raises `ValueError`
the moment it walks the second component's fields -- `dataclass` equality means the two
instances compare unequal even though their names match, and only one name can map to one fit
parameter. Wrapped in `try/except` here so the notebook keeps running; this is the deliberate,
documented failure mode, not a bug being worked around.

In [3]:
radius_a = Parameter("parent_radius", 5.0, bounds=(3.0, 7.0))
radius_b = Parameter("parent_radius", 4.2, bounds=(3.0, 7.0))  # same name, different value
assert radius_a != radius_b  # distinct dataclass instances -> unequal

try:
    DecayModel(
        channel,
        components=[
            Resonance("rho", (0, 1), RealImag(1.0, 0.0), mass=0.7753, width=0.1491, spin=1,
                      parent_radius=radius_a),
            Resonance("f0", (0, 1), RealImag(0.5, -0.2), mass=0.980, width=0.060, spin=0,
                      parent_radius=radius_b),
        ],
        normalization_method="square-dalitz", normalization_resolution=40,
    )
    raised = None
except ValueError as exc:
    raised = exc

print(f"Raised: {type(raised).__name__}: {raised}")
assert raised is not None
assert "Conflicting definitions for parameter" in str(raised)
assert "parent_radius" in str(raised)

Raised: ValueError: Conflicting definitions for parameter 'parent_radius'


## Summary

Reusing one `Parameter` instance is the sanctioned way to tie two components to a single
floatable quantity -- `DecayModel` collects it once, so Minuit sees one degree of freedom and
both components' amplitudes move together when it varies. Building two *separate* `Parameter`
objects that happen to collide on `name` is caught immediately at `DecayModel` construction
time with a `ValueError`, rather than silently letting one definition win -- a fit parameter
name is a shared namespace across the whole model, and this check is what keeps it consistent.

## Continue learning

See `dalitzplotfitter/decay.py`'s `_validate_parameters` and
[`docs/performance.md`](../../docs/performance.md) for how the prepared cache keys off
parameter identity.

Return to [the course guide](TUTORIALS.md).